# 🔎 Notebook 2 — Server-side Discovery, Health Checks, and Caching

In Notebook 1 we built a registry with heartbeats and saw **client-side** discovery.
Now we cover the rest of the pattern:

1. **Server-side discovery** — an intermediate router does the lookup so clients stay dumb.
2. **Self-registration vs third-party registration** — who puts entries in the registry?
3. **Push heartbeats vs pull health checks** — two ways the registry decides you're alive.
4. **Client-side caching** — surviving a registry outage without going dark.


## 🛠️ Setup

```bash
cd 05-microservices/service-discovery
uv sync
```

Select the `.venv` kernel in VS Code (top-right). Reload the window if the kernel doesn't appear.


## A small reusable Registry

Same idea as Notebook 1, compressed to keep the focus on routing/health-check logic.


In [1]:
import time, random

class Registry:
    def __init__(self, ttl=3.0):
        self.services, self.ttl = {}, ttl

    def register(self, name, addr):
        self.services.setdefault(name, {})[addr] = time.time()

    def heartbeat(self, name, addr):
        if addr in self.services.get(name, {}):
            self.services[name][addr] = time.time()

    def healthy(self, name):
        now = time.time()
        return [a for a, hb in self.services.get(name, {}).items() if now - hb < self.ttl]


## 1. Server-side discovery

In **client-side** discovery every client embeds the lookup + load-balancer logic. That's
fine in one language, painful in a polyglot fleet. **Server-side** discovery moves that logic
into an intermediary — a load balancer, API gateway, or Kubernetes Service. The client calls
one fixed endpoint; the router does the rest.

```
client ──HTTP──▶ router ──asks──▶ registry
                     │
                     └──HTTP──▶ chosen instance
```


In [2]:
class ServerSideRouter:
    """Clients call us at a fixed address; we pick a healthy instance."""
    def __init__(self, registry):
        self.r = registry
        self.idx = 0

    def route(self, service, path):
        instances = self.r.healthy(service)
        if not instances:
            return {"status": 503, "error": "no healthy instances"}
        # Round-robin — easier to reason about than random for a demo.
        self.idx = (self.idx + 1) % len(instances)
        target = instances[self.idx]
        return {"status": 200, "routed_to": target, "path": path}

reg = Registry(ttl=2.0)
for addr in ["svc-1", "svc-2", "svc-3"]:
    reg.register("users", addr)

router = ServerSideRouter(reg)
for _ in range(5):
    print(router.route("users", "/me"))


{'status': 200, 'routed_to': 'svc-2', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-3', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-1', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-2', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-3', 'path': '/me'}


### Simulate a crash — traffic shifts automatically

`svc-2` stops heart-beating. Once the TTL expires the router stops picking it. The other
two keep serving traffic. The client sees nothing unusual.


In [3]:
print("--- svc-2 crashes, only svc-1 & svc-3 keep heart-beating ---")
time.sleep(2.1)  # let every entry go stale
for addr in ["svc-1", "svc-3"]:
    reg.heartbeat("users", addr)

for _ in range(5):
    print(router.route("users", "/me"))


--- svc-2 crashes, only svc-1 & svc-3 keep heart-beating ---


{'status': 200, 'routed_to': 'svc-3', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-1', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-3', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-1', 'path': '/me'}
{'status': 200, 'routed_to': 'svc-3', 'path': '/me'}


### Client-side vs server-side — quick reference

| | Client-side | Server-side |
|--|-------------|-------------|
| Who knows the registry? | The client | A router / LB / gateway |
| Network hops per call  | 1 (after lookup) | 2 (client → router → instance) |
| Complexity lives in... | Every client library | One central router |
| Polyglot fleet         | Needs a client lib per language | Any HTTP client just works |
| Real-world examples    | Netflix Eureka + Ribbon, Spring Cloud LoadBalancer | AWS ALB/NLB, Kubernetes Service, Envoy, Consul Connect |


## 2. Self-registration vs third-party registration

**Self-registration (most common):** the instance calls `register()` on startup,
`deregister()` on shutdown, and sends heartbeats while alive.

- 👍 Simple, no extra component.
- 👎 Every service needs discovery-client code; a crashed instance can't deregister itself
  (this is why TTLs matter).

**Third-party registration:** an external *registrar* (often the orchestrator — e.g.
Kubernetes, Nomad, or a Consul sidecar) watches deployments and (de)registers instances on
behalf of the service.

- 👍 Services stay ignorant of the registry; language-agnostic.
- 👎 One more component to run and keep correct.


In [4]:
# Sketches of both styles — no networking, just the shape of the code.

class SelfRegisteringService:
    def __init__(self, name, addr, registry):
        self.name, self.addr, self.r = name, addr, registry

    def start(self):
        self.r.register(self.name, self.addr)
        print(f"[self-reg] {self.name}@{self.addr} registered itself")

    def tick(self):
        self.r.heartbeat(self.name, self.addr)

    def stop(self):
        self.r.services.get(self.name, {}).pop(self.addr, None)
        print(f"[self-reg] {self.name}@{self.addr} deregistered on shutdown")


class ThirdPartyRegistrar:
    """Imagine this is Kubernetes watching pods, or a Consul sidecar."""
    def __init__(self, registry):
        self.r = registry

    def on_instance_up(self, name, addr):
        self.r.register(name, addr)
        print(f"[registrar] observed {name}@{addr} start → registered")

    def on_instance_down(self, name, addr):
        self.r.services.get(name, {}).pop(addr, None)
        print(f"[registrar] observed {name}@{addr} stop → deregistered")


reg2 = Registry(ttl=5.0)
SelfRegisteringService("orders", "10.0.0.1:8080", reg2).start()

k8s_like = ThirdPartyRegistrar(reg2)
k8s_like.on_instance_up("orders", "10.0.0.2:8080")
k8s_like.on_instance_down("orders", "10.0.0.2:8080")


[self-reg] orders@10.0.0.1:8080 registered itself
[registrar] observed orders@10.0.0.2:8080 start → registered
[registrar] observed orders@10.0.0.2:8080 stop → deregistered


## 3. Push heartbeats vs pull health checks

Two ways to answer "is this instance still alive?":

- **Push (heartbeat)** — the instance periodically pings the registry: *"Still here!"*
  If the registry hasn't heard from it within `ttl`, it's considered dead.
  Used by Eureka, Consul (TTL checks), and what we built in Notebook 1.

- **Pull (active health check)** — the registry calls the instance's `/health` endpoint on a
  schedule. Bad response (or a timeout) → considered dead.
  Used by AWS target groups, Kubernetes readiness probes, Consul HTTP checks.

Real systems often mix both: a cheap heartbeat for liveness + a periodic HTTP probe for deeper
checks (DB reachable? queue consumer healthy?).


In [5]:
class PullHealthChecker:
    """Simulates the registry calling /health on every known instance."""
    def __init__(self, registry):
        self.r = registry

    def check(self, name, probe):
        dead = []
        for addr in list(self.r.services.get(name, {}).keys()):
            ok = probe(addr)                  # pretend HTTP GET addr/health
            if ok:
                self.r.heartbeat(name, addr)  # reuse the TTL machinery
            else:
                dead.append(addr)
        for addr in dead:
            self.r.services[name].pop(addr, None)
            print(f"[pull-check] {addr} failed → removed")


reg3 = Registry(ttl=5.0)
for a in ["a", "b", "c"]:
    reg3.register("cart", a)

# Our "probe": b is broken, others are fine.
def probe(addr):
    return addr != "b"

PullHealthChecker(reg3).check("cart", probe)
print("healthy after pull-check:", reg3.healthy("cart"))


[pull-check] b failed → removed
healthy after pull-check: ['a', 'c']


## 4. Client-side caching — staying up when the registry is down

The registry is now on the request path of every call. If it's slow or down, every service
suffers. Standard mitigation: **cache lookups client-side** and serve from cache if the
registry blinks. A stale cache is almost always better than 100% errors.


In [6]:
class CachedRegistryClient:
    def __init__(self, registry, cache_ttl=1.0):
        self.r = registry
        self.cache_ttl = cache_ttl
        self.cache = {}  # name -> (instances, fetched_at)

    def lookup(self, name):
        now = time.time()
        cached = self.cache.get(name)
        try:
            instances = self.r.healthy(name)
            self.cache[name] = (instances, now)
            return instances
        except Exception as e:
            # Registry is down — fall back to last known good list.
            if cached:
                print(f"[cache] registry down ({e}); serving stale entries")
                return cached[0]
            raise


reg4 = Registry(ttl=5.0)
reg4.register("inventory", "inv-1")
reg4.register("inventory", "inv-2")

client = CachedRegistryClient(reg4)
print("first lookup:", client.lookup("inventory"))

# Simulate a registry outage: swap .healthy for one that raises.
def broken(_):
    raise ConnectionError("registry unreachable")

reg4.healthy = broken
print("after outage:", client.lookup("inventory"))  # still works, thanks cache


first lookup: ['inv-1', 'inv-2']
[cache] registry down (registry unreachable); serving stale entries
after outage: ['inv-1', 'inv-2']


## 5. Load-balancing strategies (what the router/client *does* with the list)

Once discovery hands you a list of healthy instances, you still have to pick one.
The strategy matters — a bad one can overload a single box while others idle.

| Strategy | How it picks | When to use |
|----------|--------------|-------------|
| **Random** | `random.choice(instances)` | Trivial; good enough for uniform traffic |
| **Round-robin** | Walk the list in order | Default in Kubernetes kube-proxy, Envoy |
| **Least connections** | Pick the instance with fewest in-flight requests | Long-lived / unequal requests |
| **Weighted** | Bigger boxes get more traffic | Mixed instance sizes, canary rollouts |
| **Consistent hashing** | Hash of the request key → same instance | Stateful caches, sticky sessions |


In [7]:
# Four tiny strategies over the same healthy list.

from collections import defaultdict
import hashlib

INSTANCES = ["svc-1", "svc-2", "svc-3"]

class Random:
    def pick(self, _key=None):
        return random.choice(INSTANCES)

class RoundRobin:
    def __init__(self):
        self.i = 0
    def pick(self, _key=None):
        self.i = (self.i + 1) % len(INSTANCES)
        return INSTANCES[self.i]

class LeastConnections:
    def __init__(self):
        self.in_flight = defaultdict(int)
    def pick(self, _key=None):
        addr = min(INSTANCES, key=lambda a: self.in_flight[a])
        self.in_flight[addr] += 1
        return addr
    def done(self, addr):
        self.in_flight[addr] -= 1

class ConsistentHash:
    # "same user always hits the same pod" — useful for caches.
    def pick(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        return INSTANCES[h % len(INSTANCES)]

random.seed(0)
print("random     :", [Random().pick() for _ in range(4)])
rr = RoundRobin();       print("round-robin:", [rr.pick() for _ in range(4)])
lc = LeastConnections(); print("least-conn :", [lc.pick() for _ in range(4)])
ch = ConsistentHash()
print("hash(user=42):", ch.pick("user-42"), ch.pick("user-42"))  # deterministic
print("hash(user=7) :", ch.pick("user-7"),  ch.pick("user-7"))


random     : ['svc-2', 'svc-2', 'svc-1', 'svc-2']
round-robin: ['svc-2', 'svc-3', 'svc-1', 'svc-2']
least-conn : ['svc-1', 'svc-2', 'svc-3', 'svc-1']
hash(user=42): svc-1 svc-1
hash(user=7) : svc-3 svc-3


## 6. Graceful shutdown — deregister *before* you stop accepting traffic

Heartbeats + TTL handle *crashes*. Planned shutdowns (deploys, autoscaling) should be
cleaner: tell the registry *"I'm going away"* **first**, so no new traffic lands on you,
then finish in-flight requests, then exit. Skipping this step causes the dreaded
*"1% of requests fail every deploy"* pattern.

Typical sequence when the orchestrator sends `SIGTERM`:

1. **Deregister** from the service registry (or fail the readiness probe in K8s).
2. Wait a grace period (> one load-balancer refresh cycle) so callers stop picking you.
3. **Drain** — finish in-flight requests, stop accepting new ones.
4. Exit cleanly.


In [8]:
# Minimal Python sketch of graceful shutdown.
# In real code you'd register a signal.signal(SIGTERM, ...) handler.

class GracefulService:
    def __init__(self, name, addr, registry):
        self.name, self.addr, self.r = name, addr, registry
        self.in_flight = 0
        self.shutting_down = False

    def start(self):
        self.r.register(self.name, self.addr)

    def handle(self, req):
        if self.shutting_down:
            # Readiness would already be false; this is a belt-and-braces check.
            return {"status": 503, "error": "shutting down"}
        self.in_flight += 1
        try:
            return {"status": 200, "req": req, "served_by": self.addr}
        finally:
            self.in_flight -= 1

    def shutdown(self, grace=0.05):
        # 1. Stop advertising.
        self.r.services.get(self.name, {}).pop(self.addr, None)
        self.shutting_down = True
        # 2. Wait for callers to notice (normally seconds; tiny for the demo).
        time.sleep(grace)
        # 3. Drain — in real code you'd wait until in_flight == 0 or a deadline.
        print(f"[shutdown] {self.addr}: drained, in_flight={self.in_flight}")

reg5 = Registry(ttl=5.0)
svc = GracefulService("orders", "10.0.0.9:8080", reg5)
svc.start()
print("healthy:", reg5.healthy("orders"))
print(svc.handle("/v1/42"))
svc.shutdown()
print("healthy after shutdown:", reg5.healthy("orders"))


healthy: ['10.0.0.9:8080']
{'status': 200, 'req': '/v1/42', 'served_by': '10.0.0.9:8080'}


[shutdown] 10.0.0.9:8080: drained, in_flight=0
healthy after shutdown: []


## 🧠 Key takeaways

- **Client-side discovery** keeps the request path short but bakes discovery into every client.
- **Server-side discovery** is simpler for polyglot fleets and is what Kubernetes/ALB give you for free.
- **Heartbeats + TTL** (push) vs **active probes** (pull) are complementary — production systems use both.
- **Self-registration** is simple; **third-party registration** (orchestrator/sidecar) keeps services agnostic.
- **Load-balancing strategy** matters: round-robin is the safe default, consistent hashing for sticky caches, least-connections for uneven workloads.
- **Deregister on planned shutdown** — don't rely on the TTL to notice you're gone during a deploy.
- **Cache registry lookups on the client** so your whole system doesn't die when the registry has a bad minute.

Notebook 3 maps these ideas onto the real-world systems you'll meet on the job:
Kubernetes DNS, Consul, Eureka, and service meshes.
